# ML-JudgeBERT with pretrained multilingual encoders (XLM-R / MuRIL)
Run on **Google Colab → Runtime → Change runtime type → T4 GPU**. Uses the same grouped 5-fold protocol, JUDGEBERT augmentation and metrics as the paper, and adds late fusion with LegalDiff.

Expected time on a T4: ~10–15 min per model with 3 seeds (base-size encoders, 320 pairs).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code
Either upload `EnHi-LMP-JudgeBERT_code.zip` with the file pane on the left, or clone your GitHub repo (edit the URL).

In [ ]:
import os, zipfile
if os.path.exists('EnHi-LMP-JudgeBERT_code.zip'):
    zipfile.ZipFile('EnHi-LMP-JudgeBERT_code.zip').extractall('.')
else:
    !git clone https://github.com/<your-username>/EnHi-LMP-JudgeBERT.git
%cd EnHi-LMP-JudgeBERT

In [ ]:
!pip -q install -r requirements.txt transformers accelerate sentencepiece

## 2. XLM-R base

In [ ]:
!python src/finetune_pretrained.py --model xlm-roberta-base --epochs 5 --lr 2e-5

## 3. MuRIL base (Indian languages)

In [ ]:
!python src/finetune_pretrained.py --model google/muril-base-cased --epochs 5 --lr 2e-5

## 4. Compare with the CPU results reported in the paper

In [ ]:
import json, glob
E1 = json.load(open('results/exp1_summary.json'))['overall']
rows = [(m, v['all']['pearson'][0], v['all']['rmse'][0]) for m, v in E1.items()]
for f in glob.glob('results/pretrained_*.json'):
    for m, v in json.load(open(f)).items():
        rows.append((m, v['pearson'][0], v['rmse'][0]))
for m, r, e in sorted(rows, key=lambda x: -x[1]):
    print(f'{m:55s} r={r:.3f}  RMSE={e:.2f}')

## 5. (After human annotation) re-run on human labels
```
!python src/merge_annotations.py annotation/EnHi-LMP_annotation_A.xlsx annotation/EnHi-LMP_annotation_B.xlsx
!LMP_DATA=enhi_lmp_human.jsonl python src/finetune_pretrained.py --model xlm-roberta-base
```